# Week 2 — RAG Setup
載入財經文本 → Chunking → 建立 ChromaDB → 測試檢索

In [1]:
import sys
sys.path.insert(0, '..')
from src.rag.loader import load_all_documents
from src.rag.retriever import build_vectordb, load_vectordb, retrieve

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 1 — 載入文本資料

In [2]:
tickers = ['0050.TW', '2330.TW', '2454.TW', '2317.TW']
documents = load_all_documents(tickers)
print(f'\nTotal documents loaded: {len(documents)}')
print('\nSample document:')
print(documents[0].page_content[:300])
print('Metadata:', documents[0].metadata)

Loading yfinance news for ['0050.TW', '2330.TW', '2454.TW', '2317.TW']...


  Loaded 30 news articles
Total documents: 30

Total documents loaded: 30

Sample document:
Title: Elon Musk's SpaceX submits plans for $55B Terafab chip facility
Summary: Elon Musk's SpaceX (SPAX.PVT) has filed plans to invest $55 billion into building a Terafab facility in Texas, which will allow the company to produce in-house chips for its AI and robotics projects. Yahoo Finance Senior
Metadata: {'source': 'yfinance_news', 'ticker': '2330.TW', 'date': '2026-05-06T20:41:47Z'}


## Task 2 — 建立向量資料庫

In [3]:
# Build and persist ChromaDB (takes a few minutes on first run)
vectordb = build_vectordb(documents, chunk_size=512, chunk_overlap=64)
print('Vector DB ready.')

Split into 39 chunks


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 20135.21it/s]

Vector DB saved to C:\Data_science\Final\notebooks\..\data\vectordb
Vector DB ready.


## Task 3 — 檢索功能測試

In [4]:
# Test queries
test_queries = [
    '台積電最近的營收表現如何？',
    '0050 ETF 的投資策略是什麼？',
    '半導體產業的前景',
]

for query in test_queries:
    print(f'\n--- Query: {query} ---')
    results = retrieve(query, vectordb, k=2)
    for i, doc in enumerate(results, 1):
        print(f'[{i}] {doc.page_content[:200]}...')
        print(f'    Source: {doc.metadata.get("source")} | Ticker: {doc.metadata.get("ticker", "N/A")}')


--- Query: 台積電最近的營收表現如何？ ---
[1] Title: Lantronix Q3 Earnings Call Highlights
Summary: Lantronix (NASDAQ:LTRX) reported fiscal third-quarter results that were in line with management’s outlook, as the company pointed to continued mom...
    Source: yfinance_news | Ticker: 2454.TW
[2] Title: Hon Hai Reports 29.7% Revenue Jump As AI Hardware Demand Holds
Summary: April revenue reached NT$832.1 billion as Hon Hai expects second-quarter sales to grow sequentially and year-over-year....
    Source: yfinance_news | Ticker: 2317.TW

--- Query: 0050 ETF 的投資策略是什麼？ ---
[1] Summary: The semiconductor sector continues to absorb capital at a pace tied to the AI infrastructure buildout, and three exchange-traded funds offer distinct angles on it: iShares Semiconductor ETF (...
    Source: yfinance_news | Ticker: 2330.TW
[2] Summary: First it was FAANG, and then it was the Magnificent Seven. Are investors ready for the AIR 7 grouping of tech stocks? Hennion & Walsh CIO Kevin Mahn joins Market Domina

## Load existing DB (跳過重建)

In [5]:
# Next time, load existing DB directly without rebuilding
vectordb = load_vectordb()
results = retrieve('台積電股價', vectordb)
print(f'Retrieved {len(results)} documents')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19555.15it/s]

Retrieved 4 documents


C:\Data_science\Final\notebooks\..\src\rag\retriever.py:32: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(
